# Gold : `dim_customers`

**Entrées :** `workspace.silver.orders`, `workspace.silver.customers`
**Sortie :** `workspace.gold.dim_customers` (**une ligne par personne**, clé : `customer_unique_id`)

## Ce qui change de Silver vers Gold
| Silver | Gold |
|---|---|
| `customers` : une ligne **par commande** (99 441) | `dim_customers` : une ligne **par personne** (96 096) |
| Adresse attachée à chaque commande | Une seule localisation par personne : celle de **sa commande la plus récente** (décision à valider) |
| Aucun calcul | `order_count`, `sale_count`, première et dernière commande |

La colonne est nommée « dernière localisation de **livraison** » et non « adresse » : d'après la documentation de la source, `customers` donne le lieu de livraison de la commande. L'adresse de chaque commande reste dans `fact_orders`.

**Définition provisoire :** une vente est une commande ni `canceled` ni `unavailable`.

**Départage :** si deux commandes ont la même date d'achat, on prend celle dont l'`order_id` est le plus grand (pour que le résultat soit toujours le même).

In [0]:
CREATE OR REPLACE TABLE workspace.gold.dim_customers AS
WITH commandes AS (
    SELECT
        o.customer_unique_id,
        o.order_id,
        o.order_status,
        o.order_purchase_ts,
        c.customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,
        ROW_NUMBER() OVER (
            PARTITION BY o.customer_unique_id
            ORDER BY o.order_purchase_ts DESC, o.order_id DESC
        ) AS rang
    FROM workspace.silver.silver_orders o
    JOIN workspace.silver.silver_customers c ON o.customer_id = c.customer_id
)
SELECT
    customer_unique_id,
    COUNT(*) AS order_count,
    COUNT(*) FILTER (WHERE order_status NOT IN ('canceled', 'unavailable')) AS sale_count,
    MIN(order_purchase_ts) AS first_order_at,
    MAX(order_purchase_ts) AS last_order_at,
    MAX(CASE WHEN rang = 1 THEN customer_zip_code_prefix END) AS last_delivery_zip_prefix,
    MAX(CASE WHEN rang = 1 THEN customer_city END)            AS last_delivery_city,
    MAX(CASE WHEN rang = 1 THEN customer_state END)           AS last_delivery_state
FROM commandes
GROUP BY customer_unique_id

num_affected_rows,num_inserted_rows


## Vérification
| Mesure | Attendu |
|---|---:|
| `personnes`, `personnes_distinctes` | 96 096 |
| `commandes_total` | 99 441 (aucune commande perdue) |
| `personnes_reachat` | 2 997 |
| `sans_localisation` | 0 |

In [0]:
SELECT
    COUNT(*) AS personnes,
    COUNT(DISTINCT customer_unique_id) AS personnes_distinctes,
    SUM(order_count) AS commandes_total,
    COUNT(*) FILTER (WHERE order_count > 1) AS personnes_reachat,
    COUNT(*) FILTER (WHERE last_delivery_state IS NULL) AS sans_localisation
FROM workspace.gold.dim_customers

personnes,personnes_distinctes,commandes_total,personnes_reachat,sans_localisation
96096,96096,99441,2997,0
